# Etapa 4 — Imagen del dinosaurio (Colab con GPU)

Entorno: **Entorno de ejecución → Cambiar tipo → GPU T4**.

Modelo text-to-image: `amused/amused-512` (aMUSEd, ~800 M parámetros, 512×512, 12 pasos).
El prompt se arma con el nombre del modelo de caracteres y los rasgos que devolvió Ollama (etapa 3).

In [ ]:
%pip install -q diffusers transformers accelerate safetensors torch

In [ ]:
import torch, time
from diffusers import AmusedPipeline

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe = AmusedPipeline.from_pretrained('amused/amused-512', variant='fp16', torch_dtype=torch.float16).to(dispositivo)
pipe.vqvae.to(torch.float32)  # el VQ-GAN en fp16 produce artefactos
print(dispositivo, torch.cuda.get_device_name(0) if dispositivo == 'cuda' else '')

## Prompt
Nombre de la etapa 2 + `prompt_imagen` generado por Ollama en la etapa 3 (ver `2_ollama_sagemaker/identidad_seleccionado.json`).

In [ ]:
import json, urllib.request

NOMBRE = 'Limedsaurus'
# Campo prompt_imagen de 2_ollama_sagemaker/identidad_limedsaurus.json
RASGOS = 'Photorealistic depiction of a large, gray Limedsaurus with a prominent bony crest, stalking through a misty swamp habitat. Nature documentary style.'

prompt = f'{NOMBRE}, a dinosaur. {RASGOS}'
negativo = 'blurry, low quality, text, watermark, deformed, extra limbs, cartoon, cropped'
print(prompt)

In [ ]:
g = torch.Generator(dispositivo).manual_seed(7)
t0 = time.time()
img = pipe(prompt, negative_prompt=negativo, num_inference_steps=12, guidance_scale=10.0, generator=g).images[0]
print(f'{time.time() - t0:.1f} s')
img.save(f'{NOMBRE.lower()}.png')
img

## (Opcional) Exponer la generación como servicio
La app desplegada usa el mismo `servidor_imagen.py` corriendo en el notebook de SageMaker (CPU, ~60 s por imagen). Si se quiere usar la GPU de Colab, se puede levantar el servidor aquí y publicarlo con ngrok **usando tu propio token, que no se sube al repositorio**:

In [ ]:
# !wget -q https://raw.githubusercontent.com/<usuario>/<repo>/main/3_imagen/servidor_imagen.py
# %pip install -q fastapi uvicorn pyngrok
# import subprocess; subprocess.Popen(['uvicorn', 'servidor_imagen:app', '--port', '8000'])
# from pyngrok import ngrok; from google.colab import userdata
# ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
# print(ngrok.connect(8000).public_url)  # POST <url>/generar {"prompt": ...}